In [2]:
import pandas as pd
import numpy as np


In [3]:
df = pd.read_csv('../data/data.csv')
df.head()

,symbol,company_name,stock_id,date,high,low,close,volume
0,AAF.N0000,NaN,2025,2025-02-23,28.4,27.4,27.6,17164
1,AAF.N0000,NaN,2025,2025-02-24,27.7,27.0,27.1,7199
2,AAF.N0000,NaN,2025,2025-02-26,27.6,26.9,27.1,10046
3,AAF.N0000,NaN,2025,2025-02-27,28.0,27.5,27.7,1681
4,AAF.N0000,NaN,2025,2025-03-02,28.4,26.3,26.7,7884


In [7]:
arr = np.array(df['symbol'].str.split('.').tolist())
arr[:,0]
df['symbol'] = arr

In [8]:
df['symbol'].nunique()

274

# Mapping sectors
## 20 GICS (Global Industry Classification Standard)

1.Energy
2.Materials
3.Capital Goods
4.Commercial and Professional Services
5.Transportation
6.Automobiles and Components
7.Consumer Durables and Apparel
8.Consumer Services
9.Retailing
10.Food and Staples Retailing
11.Food, Beverage and Tobacco
12.Household and Personal Products
13.Health Care Equipment and Services
14.Pharmaceuticals, Biotechnology and Life Sciences
15.Banks
16.Diversified Financials
17.Insurance
18.Telecommunication Services
19.Utilities
20.Real Estate


#Using claude opus 4.6 model 

In [11]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()


client = OpenAI(
    api_key=os.getenv('OPENROUTER_API_KEY'),
    base_url="https://openrouter.ai/api/v1",
)
def pred_sector(name: str):

    prompt = f""" 

        There are companies of 20 sectors listed in colombo stock exchange.Sectors are
        Energy, Materials, Capital Goods, Commercial and Professional Services, Transportation,
        Automobiles and Components,Consumer Durables and Apparel, Consumer Services, Retailing, 
        Food and Staples Retailing, Food , Beverage and Tobacco, Household and Personal Products, 
        Health Care Equipment and Services, Pharmaceuticals, Biotechnology and Life Sciences, Banks,
        Diversified Financials,Insurance, Telecommunication Services, Utilities, Real Estate.
        '{name}' is the symbol of the company listed in colombo stock exchange Sri Lanka.
        take the name of the company and search which sector the company falls into from the above list.

        Your response only should contain attributes from the list above without any additional characters.
        """

    response = client.chat.completions.create(
    model="anthropic/claude-opus-4.6",
    messages=[
        {
            "role": "system",
            "content": "You are a financial analyst specializing in the Colombo Stock Exchange. You classify companies into their correct CSE sectors. You respond with only the sector name, nothing else."
        },
        {"role": "user", "content": prompt}
    ]
    )
    return response.choices[0].message.content.strip()



In [ ]:
names = df['symbol'].unique().tolist()
map = {name : pred_sector(name) for name in names }
df['sector'] = df['symbol']
df['sector'] = df['sector'].map(map)


In [49]:
df.head()

,symbol,company_name,stock_id,date,high,low,close,volume,suffix,sector
0,AAF,NaN,2025,2025-02-23,28.4,27.4,27.6,17164,N0000,Diversified Financials
1,AAF,NaN,2025,2025-02-24,27.7,27.0,27.1,7199,N0000,Diversified Financials
2,AAF,NaN,2025,2025-02-26,27.6,26.9,27.1,10046,N0000,Diversified Financials
3,AAF,NaN,2025,2025-02-27,28.0,27.5,27.7,1681,N0000,Diversified Financials
4,AAF,NaN,2025,2025-03-02,28.4,26.3,26.7,7884,N0000,Diversified Financials


In [69]:
df['sector'].unique().tolist()

['Diversified Financials',
 'Food , Beverage and Tobacco',
 'Materials',
 'Capital Goods',
 'Insurance',
 'Health Care Equipment and Services',
 'Automobiles and Components',
 'Consumer Durables and Apparel',
 'Banks',
 'Consumer Services',
 'Real Estate',
 'Food and Staples Retailing',
 'Transportation',
 'Household and Personal Products',
 'Utilities',
 'Energy',
 'Retailing',
 'Pharmaceuticals, Biotechnology and Life Sciences',
 'Commercial and Professional Services',
 'Telecommunication Services',
 'Beverages, Food and Tobacco\n\nWait, let me reconsider the exact sector names from the list.\n\nFood , Beverage and Tobacco']

In [ ]:
df[df['sector'] == 'Please provide the company name or symbol so I can classify it into the correct CSE sector.'] = 'Diversified Financials'
df[df['sector'] == 'Beverages, Food and Tobacco\n\nWait, let me reconsider the exact sector names from the list.\n\nFood , Beverage and Tobacco']['symbol'].unique()
df[df['sector'] == 'Beverages, Food and Tobacco\n\nWait, let me reconsider the exact sector names from the list.\n\nFood , Beverage and Tobacco'] = 'Automobiles and Components'


/var/folders/db/4m8dhthn51d8sxyp9mp9sd_00000gn/T/ipykernel_71868/67252455.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Diversified Financials' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df[df['sector'] == 'Please provide the company name or symbol so I can classify it into the correct CSE sector.'] = 'Diversified Financials'
/var/folders/db/4m8dhthn51d8sxyp9mp9sd_00000gn/T/ipykernel_71868/67252455.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Diversified Financials' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df[df['sector'] == 'Please provide the company name or symbol so I can classify it into the correct CSE sector.'] = 'Diversified Financials'
/var/folders/db/4m8dhthn51d8sxyp9mp9sd_00000gn/T/ipykernel_71868/6725245

In [3]:
df.drop('company_name',axis = 1,inplace = True)
df.to_csv('../output/sector-llm-imputed.csv')

NameError: name 'df' is not defined